In [1]:
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd


In [2]:
url = "https://www.finhacker.cz/en/top-20-sp-500-companies-by-market-cap/"
html = requests.get(url, timeout=30).text

soup = BeautifulSoup(html, "html.parser")

In [3]:
details_blocks = soup.find_all("details", attrs={"data-year": True})

In [4]:
def parse_market_cap(value: str) -> float:
    value = value.strip().upper()

    if value.endswith("T"):
        return float(value[:-1]) * 1e12
    if value.endswith("B"):
        return float(value[:-1]) * 1e9
    if value.endswith("M"):
        return float(value[:-1]) * 1e6

    raise ValueError(f"Formato Market Cap sconosciuto: {value}")


In [5]:
ticker_cache = {}

def extract_ticker(company_link: str) -> str | None:
    if not company_link:
        return None
    url = "https://www.finhacker.cz" + company_link

    
    if company_link in ticker_cache:
            return ticker_cache[company_link]


    try:
        html = requests.get(url, timeout=30).text
    except Exception:
        return None

    soup = BeautifulSoup(html, "html.parser")
    scripts = soup.find_all("script")

    for script in scripts:
        if not script.string:
            continue

        match = re.search(
            r"ticker=([A-Z0-9\-\.]+)",
            script.string
        )
        if match:
            ticker = match.group(1)
            ticker_cache[company_link] = ticker
            return ticker

    return None

In [6]:
def normalize_sector(css_class: str) -> str | None:
    if not css_class:
        return None

    return " ".join(word.capitalize() for word in css_class.split("-"))

In [7]:
records = []

for block in details_blocks:
    year = int(block["data-year"])
    print(f"Lavoro anno {year}")

    table = block.find("table")
    if table is None:
        continue

    rows = table.find("tbody").find_all("tr")

    for row in rows:
        tds = row.find_all("td")
        if len(tds) < 4:
            continue  # sicurezza

        # Rank
        rank = int(tds[0].get_text(strip=True))

        # Company name + link
        company_cell = tds[1]
        strong = company_cell.find("strong")
        company_name = strong.get_text(strip=True)

        link_tag = company_cell.find("a", href=True)
        company_link = link_tag["href"] if link_tag else None

        ticker = extract_ticker(company_link)

        # Market cap
        market_cap_raw = tds[2].get_text(strip=True)
        market_cap = parse_market_cap(market_cap_raw)

        # Sector (classe del div.bar)
        sector = None
        bar_div = tds[3].find("div", class_="bar")
        if bar_div and bar_div.has_attr("class"):
            classes = bar_div["class"]
            sector_classes = [c for c in classes if c != "bar"]
            sector = sector_classes[0] if sector_classes else None
            sector = normalize_sector(sector)

        records.append({
            "Year": year,
            "Rank": rank,
            "Ticker": ticker,
            "Company": company_name,
            "Sector": sector,
            "Market Cap": market_cap
        })

Lavoro anno 2026
Lavoro anno 2025
Lavoro anno 2024
Lavoro anno 2023
Lavoro anno 2022
Lavoro anno 2021
Lavoro anno 2020
Lavoro anno 2019
Lavoro anno 2018
Lavoro anno 2017
Lavoro anno 2016
Lavoro anno 2015
Lavoro anno 2014
Lavoro anno 2013
Lavoro anno 2012
Lavoro anno 2011
Lavoro anno 2010
Lavoro anno 2009
Lavoro anno 2008
Lavoro anno 2007
Lavoro anno 2006
Lavoro anno 2005
Lavoro anno 2004
Lavoro anno 2003
Lavoro anno 2002
Lavoro anno 2001
Lavoro anno 2000
Lavoro anno 1999
Lavoro anno 1998
Lavoro anno 1997
Lavoro anno 1996
Lavoro anno 1995
Lavoro anno 1994
Lavoro anno 1993
Lavoro anno 1992
Lavoro anno 1991
Lavoro anno 1990
Lavoro anno 1989


In [8]:
df_dettaglio = (
    pd.DataFrame(records)
    .sort_values(["Year", "Rank"])
    .reset_index(drop=True)
)

df_dettaglio.head()


,Year,Rank,Ticker,Company,Market Cap,Sector
0,1989,1,XOM,Exxon Mobil,6.259000e+10,Energy
1,1989,2,IBM,IBM,5.203000e+10,Technology
2,1989,3,GE,General Electric,3.578000e+10,Industrials
3,1989,4,BMY,Bristol-Myers Squibb,2.800000e+10,Health Care
4,1989,5,MRK,Merck,2.758000e+10,Health Care


In [9]:
df_dettaglio.to_csv(
    "data/sp500top20.csv",
    index=False
)

In [10]:
totals_records = []

for block in details_blocks:
    year = int(block["data-year"])
    print(f"Lavoro anno {year}")
    
    table = block.find("table")
    if table is None:
        continue

    tfoot = table.find("tfoot")
    if tfoot is None:
        continue

    cap_tds = tfoot.find_all("td", class_="cap")

    if len(cap_tds) < 2:
        continue  # sicurezza

    # ordine garantito dal DOM
    top20_raw = cap_tds[0].get_text(strip=True)
    sp500_raw = cap_tds[1].get_text(strip=True)

    top20_cap = parse_market_cap(top20_raw)
    sp500_cap = parse_market_cap(sp500_raw)

    totals_records.append({
        "Year": year,
        "Top20 Market Cap": top20_cap,
        "SP500 Market Cap": sp500_cap
    })

Lavoro anno 2026
Lavoro anno 2025
Lavoro anno 2024
Lavoro anno 2023
Lavoro anno 2022
Lavoro anno 2021
Lavoro anno 2020
Lavoro anno 2019
Lavoro anno 2018
Lavoro anno 2017
Lavoro anno 2016
Lavoro anno 2015
Lavoro anno 2014
Lavoro anno 2013
Lavoro anno 2012
Lavoro anno 2011
Lavoro anno 2010
Lavoro anno 2009
Lavoro anno 2008
Lavoro anno 2007
Lavoro anno 2006
Lavoro anno 2005
Lavoro anno 2004
Lavoro anno 2003
Lavoro anno 2002
Lavoro anno 2001
Lavoro anno 2000
Lavoro anno 1999
Lavoro anno 1998
Lavoro anno 1997
Lavoro anno 1996
Lavoro anno 1995
Lavoro anno 1994
Lavoro anno 1993
Lavoro anno 1992
Lavoro anno 1991
Lavoro anno 1990
Lavoro anno 1989


In [11]:
df_totale = (
    pd.DataFrame(totals_records)
    .sort_values("Year")
    .reset_index(drop=True)
)

df_totale.head()


,Year,Top20 Market Cap,SP500 Market Cap
0,1989,4.640000e+11,2.300000e+12
1,1990,5.070000e+11,2.230000e+12
2,1991,6.690000e+11,2.910000e+12
3,1992,6.580000e+11,3.130000e+12
4,1993,6.720000e+11,3.450000e+12


In [12]:
df_totale.to_csv(
    "data/sp500cap.csv",
    index=False
)